In [ ]:
# Cell 1 — Install dependencies
# Requires A100 (or any GPU) for Unsloth GRPO training.
# Pin unsloth to the version tested with this codebase.
!pip install -q \
    "unsloth==2025.3.19" \
    "trl>=0.15.0" \
    "openenv-core" \
    "pydantic>=2.0" \
    "pyyaml" \
    "datasets" \
    "httpx"
print('Install complete.')

In [ ]:
# Cell 2 — Clone repo and install the package in editable mode
import os

REPO_URL = "https://github.com/YOUR_ORG/enterprise-ops-arena.git"  # TODO: set your repo URL
REPO_DIR = "enterprise-ops-arena"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .
print('Repo ready.')

In [ ]:
# Cell 3 — Run GRPO training
# Trains Qwen2.5-3B-Instruct with Unsloth 4-bit + GRPO for 200 steps.
# Logs metrics to logs/metrics.csv  every 10 steps.
# Saves LoRA adapters to checkpoints/  every 50 steps.
!python -m enterprise_ops.train.main \
    --scenario scenario_01 \
    --steps 200 \
    --episode-length 8 \
    --grpo-gens 4 \
    --log-every 10 \
    --save-every 50 \
    --log-dir logs \
    --checkpoint-dir checkpoints

In [ ]:
# Cell 4 — Plot reward curves from logs/metrics.csv
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

df = pd.read_csv("logs/metrics.csv")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("EnterpriseOps Arena — GRPO Training Reward Curves", fontsize=14, fontweight="bold")

SERIES = [
    ("episode_score",      "Total Episode Score",      "tab:blue",   axes[0, 0]),
    ("task_completion",    "Task Completion",           "tab:green",  axes[0, 1]),
    ("sla_rate",           "SLA Adherence Rate",        "tab:orange", axes[1, 0]),
    ("coordination_score", "Coordination Score",        "tab:purple", axes[1, 1]),
]

for col, title, color, ax in SERIES:
    ax.plot(df["step"], df[col], color=color, linewidth=1.8, label=col)
    # Rolling mean
    win = max(1, len(df) // 10)
    ax.plot(
        df["step"], df[col].rolling(win, min_periods=1).mean(),
        color=color, linewidth=2.5, linestyle="--", alpha=0.85,
        label=f"{win}-step avg",
    )
    # Shade curriculum difficulty changes
    if "curriculum_difficulty" in df.columns:
        diffs = df["curriculum_difficulty"].diff().fillna(0)
        for idx in df.index[diffs != 0]:
            ax.axvline(df.loc[idx, "step"], color="gray", linestyle=":", alpha=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Training step")
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("logs/reward_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: logs/reward_curves.png")

# Summary table
print("\n--- Final metrics ---")
print(df.tail(5).to_string(index=False))